# Tunix-Med: SFT Fine-Tuning with JAX/Tunix

Fine-tunes `google/gemma-3-1b-it` on a Medical QA dataset using **Tunix** (LoRA, bfloat16, streaming data).

## 1 · Environment & Logging Setup

Silence the absl / JAX / Orbax / TensorFlow noise **before** any heavy imports so nothing slips through.

In [1]:
import os
import sys
import logging

# ── JAX memory: do NOT pre-allocate the full 75% pool ────────────────────────
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

# ── silence verbose back-ends ────────────────────────────────────────────────
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"  # TensorFlow C++ logs
os.environ["GRPC_VERBOSITY"] = "ERROR"
os.environ["GLOG_minloglevel"] = "3"
os.environ["JAX_LOGGING_LEVEL"] = "ERROR"

# Python-level loggers
for _name in (
    "absl",
    "jax",
    "orbax",
    "flax",
    "datasets",
    "huggingface_hub",
    "transformers",
    "grain",
):
    logging.getLogger(_name).setLevel(logging.ERROR)
logging.basicConfig(level=logging.ERROR, stream=sys.stdout)

# absl logger (used by Orbax/Tunix internals)
import absl.logging

absl.logging.set_verbosity(absl.logging.ERROR)

# ── JAX ──────────────────────────────────────────────────────────────────────
import jax
import jax.numpy as jnp
import numpy as np

try:
    if jax.process_count() > 1:
        jax.distributed.initialize()
except Exception:
    pass  # single-host run

print(f"JAX backend : {jax.default_backend()}")
print(f"Devices     : {jax.devices()}")

JAX backend : gpu
Devices     : [CudaDevice(id=0)]


## 2 · Hyperparameter Config

In [2]:
class Config:
    # ── model ─────────────────────────────────────────────────────────────────
    MODEL_NAME = "google/gemma-3-1b-it"
    MODEL_KEY  = "gemma-3-1b-it"       # key expected by tunix automodel

    # ── data ──────────────────────────────────────────────────────────────────
    DATASET_PATH = "data/medical-cardiology-qa"
    MAX_SEQ_LEN  = 1024               # fixed length — avoids JIT recompilation
    EVAL_SPLIT   = 0.1
    SEED         = 42

    # ── LoRA ─────────────────────────────────────────────────────────────────
    LORA_RANK    = 16
    LORA_ALPHA   = 32    # ← back to run 1 (scale=2.0)
    LORA_DROPOUT = 0.05  # keep — this is still useful regularisation

    # ── optimiser ────────────────────────────────────────────────────────────
    LEARNING_RATE = 2e-4  # ← back to run 1 — it found the deepest minimum
    WEIGHT_DECAY  = 0.01  # ← back to run 1
    WARMUP_RATIO  = 0.05

    # ── training ─────────────────────────────────────────────────────────────
    BATCH_SIZE          = 4
    GRAD_ACCUM_STEPS    = 4
    NUM_EPOCHS          = 1   # ← best was always found in epoch 1
    EARLY_STOP_PATIENCE = 5   # ← new: stop if no improvement for 5 evals
    MAX_STEPS           = None
    EVAL_EVERY_N_STEPS  = 250
    LOG_EVERY_N_STEPS   = 250

    # ── output ────────────────────────────────────────────────────────────────
    OUTPUT_DIR = "tunix-medical-model"


params = Config()
print("Config loaded.")
print(f"  Effective batch size : {params.BATCH_SIZE * params.GRAD_ACCUM_STEPS}")
print(f"  Max seq length       : {params.MAX_SEQ_LEN}")
print(f"  LoRA rank / alpha    : {params.LORA_RANK} / {params.LORA_ALPHA}")
print(f"  Warmup ratio         : {params.WARMUP_RATIO}")


Config loaded.
  Effective batch size : 16
  Max seq length       : 1024
  LoRA rank / alpha    : 16 / 32
  Warmup ratio         : 0.05


## 3 · Hugging Face Authentication

In [3]:
from huggingface_hub import login

hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    login(token=hf_token, add_to_git_credential=False)
    print("Logged in to Hugging Face.")
else:
    print("No HF_TOKEN found — set it if the model is gated.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Logged in to Hugging Face.


## 4 · Streaming Data Pipeline

**Why this avoids the 91 GB spike:**
- The HuggingFace `Dataset` object keeps data **on disk** in Arrow format; only the rows requested per batch are deserialised.
- Tokenisation happens lazily inside the generator — no pre-tokenised tensor cache is ever materialised in RAM.
- Every batch is padded to the **same fixed length** (`MAX_SEQ_LEN`). This means JAX traces the `train_step` function exactly once and reuses the compiled XLA kernel every subsequent step — the main reason GPU utilisation was low before.

In [4]:
import datasets
from transformers import AutoTokenizer
from tunix.sft import peft_trainer

# ── tokeniser ────────────────────────────────────────────────────────────────
print("Loading tokeniser …")
hf_tokenizer = AutoTokenizer.from_pretrained(params.MODEL_NAME)

# ── dataset — keep ONE memory-mapped reference ────────────────────────────────
# load_from_disk returns an Arrow-backed Dataset; individual row access is lazy.
# We must NOT call train_test_split(): it writes two new Arrow files, which
# materialises the entire dataset in RAM (the source of the 91 GB spike).
print("Loading dataset metadata …")
full_ds = datasets.load_from_disk(params.DATASET_PATH)["train"]
n = len(full_ds)

# Pre-compute split index arrays — only integers live in RAM, not the data.
rng = np.random.default_rng(params.SEED)
all_idx = rng.permutation(n)
cut = int(n * (1.0 - params.EVAL_SPLIT))
train_idx = all_idx[:cut]  # shape (N_train,) of int64 — negligible memory
eval_idx = all_idx[cut:]  # shape (N_eval,)  of int64

print(f"  Total rows : {n:,}")
print(f"  Train rows : {len(train_idx):,}")
print(f"  Eval  rows : {len(eval_idx):,}")


# ── per-example tokenisation ─────────────────────────────────────────────────
def _tokenise(example: dict) -> dict:
    """Convert one messages-style example into fixed-length token + mask arrays."""
    messages = example["messages"]
    
    # Use tokenize=True to ensure correct BOS handling and formatting
    full_ids = hf_tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=False
    )
    # Get prompt length by tokenising everything except the last (assistant) message
    prompt_ids = hf_tokenizer.apply_chat_template(
        messages[:-1], tokenize=True, add_generation_prompt=True
    )
    
    prompt_len = len(prompt_ids)
    full_ids = full_ids[: params.MAX_SEQ_LEN]
    
    mask = np.ones(len(full_ids), dtype=np.int32)
    mask[: min(prompt_len, len(full_ids))] = 0

    pad_len = params.MAX_SEQ_LEN - len(full_ids)
    tokens = np.pad(full_ids, (0, pad_len), constant_values=hf_tokenizer.pad_token_id).astype(np.int32)
    mask = np.pad(mask, (0, pad_len), constant_values=0).astype(np.int32)
    return {"input_tokens": tokens, "input_mask": mask}


# ── streaming batch generator ─────────────────────────────────────────────────
def make_data_iterator(
    dataset,
    index_array: np.ndarray,  # pre-computed split indices — just ints
    batch_size: int,
    shuffle: bool = True,
    infinite: bool = True,
):
    """
    Reads rows from the Arrow store one at a time via dataset[int(i)].
    No full-dataset copy is ever created in memory.
    """
    while True:
        idxs = index_array.copy()
        if shuffle:
            np.random.shuffle(idxs)
        for start in range(0, len(idxs) - batch_size + 1, batch_size):
            batch_idx = idxs[start : start + batch_size]
            processed = [_tokenise(dataset[int(i)]) for i in batch_idx]
            tokens = np.stack([p["input_tokens"] for p in processed])
            masks = np.stack([p["input_mask"] for p in processed])
            yield peft_trainer.TrainingInput(
                input_tokens=jnp.array(tokens),
                input_mask=jnp.array(masks),
            )
        if not infinite:
            break


train_iter = make_data_iterator(
    full_ds, train_idx, params.BATCH_SIZE, shuffle=True, infinite=True
)
eval_iter = make_data_iterator(
    full_ds, eval_idx, params.BATCH_SIZE, shuffle=False, infinite=False
)

# ── sanity check ──────────────────────────────────────────────────────────────
sample = next(
    make_data_iterator(full_ds, train_idx[:2], 2, shuffle=False, infinite=False)
)
print(
    f"\nBatch shape : tokens={sample.input_tokens.shape}  mask={sample.input_mask.shape}"
)
print(f"Dtype       : {sample.input_tokens.dtype}")
print("Data pipeline ready.")

Loading tokeniser …
Loading dataset metadata …
  Total rows : 10,518
  Train rows : 9,466
  Eval  rows : 1,052

Batch shape : tokens=(2, 1024)  mask=(2, 1024)
Dtype       : int32
Data pipeline ready.


## 5 · Model + LoRA Setup

The model is loaded in **bfloat16** (`~2 GB` weights for Gemma 3-1B) instead of float32 (~4 GB). Combined with the fixed-sequence data pipeline, total HBM usage drops from ~91 GB to roughly 4–6 GB on a single GPU.

In [5]:
import shutil
from jax.sharding import Mesh
from flax import nnx
from tunix.models import automodel
from tunix.sft import utils as sft_utils
from tunix.sft.utils import show_hbm_usage
from huggingface_hub import snapshot_download

# ── device mesh ───────────────────────────────────────────────────────────────
devices = np.array(jax.devices()).reshape((1, -1))
mesh    = Mesh(devices, axis_names=("tp", "fsdp"))
print(f"Mesh shape : {devices.shape}  axes=(tp, fsdp)")

# ── download model weights ────────────────────────────────────────────────────
print("Downloading / verifying model weights …")
resolved_model_path = snapshot_download(params.MODEL_NAME)
print(f"Weights at  : {resolved_model_path}")

# ── build sharded model in bfloat16 ──────────────────────────────────────────
print("Building model (bfloat16) …")
model_config = automodel.call_model_config(params.MODEL_KEY)
with mesh:
    model = automodel.create_model_from_safe_tensors(
        params.MODEL_KEY,
        resolved_model_path,
        model_config,
        mesh,
        dtype=jnp.bfloat16,
    )


# ── LoRA application ──────────────────────────────────────────────────────────
# Gemma 3 attention layers use nnx.Einsum (not nnx.Linear), so they are NOT
# found by isinstance(module, nnx.Linear).  Without attention LoRA the model
# can only shift MLP "knowledge representations" but cannot change how it
# ATTENDS to the question — critical for QA performance.
#
# We add a lightweight custom LoRA wrapper for the attention Einsum layers.
# The standard LoRA formulas still apply; we just reshape the einsum weight
# to a 2-D matrix, apply low-rank decomposition, then reshape back.

class EinsumLoRALayer(nnx.Module):
    """
    LoRA wrapper for nnx.Einsum modules found in Gemma 3 attention.
    Stores the frozen original weight plus two trainable LoRA matrices A and B.
    Forward:  y = original_einsum(x) + lora_scale * einsum_lora(dropout(x), A, B)
    """
    def __init__(self, einsum_module, rank: int, alpha: float, dropout: float,
                 dtype=jnp.bfloat16, rngs=None):
        self.einsum   = einsum_module          # frozen original
        self.rank     = rank
        self.scale    = alpha / rank
        self.dtype    = dtype
        self.dropout  = nnx.Dropout(dropout, rngs=rngs)
        w = einsum_module.w
        
        
        
        
                      # original weight tensor
        self.w_shape  = w.shape
        # Flatten to 2-D: (product of all dims except last) x last_dim
        in_dim  = 1
        for d in w.shape[:-1]: in_dim *= d
        out_dim = w.shape[-1]
        # A: (in_dim, rank)  initialised with small normal
        # B: (rank, out_dim) initialised with zeros → no initial delta
        key_a, key_b = jax.random.split(rngs.params() if rngs else jax.random.PRNGKey(0))
        a_val = jax.random.normal(key_a, (in_dim, rank), dtype=dtype) * 0.01
        b_val = jnp.zeros((rank, out_dim), dtype=dtype)
        self.lora_a = nnx.LoRAParam(a_val)
        self.lora_b = nnx.LoRAParam(b_val)

    def __call__(self, x):
        base = self.einsum(x)
        # Apply dropout to the input for the LoRA path
        lora_x = self.dropout(x)
        # Reshape weight-space perturbation: A @ B then reshape to w_shape
        delta_w_flat = self.lora_a[...] @ self.lora_b[...]  # (in_dim, out_dim)
        delta_w      = delta_w_flat.reshape(self.w_shape)
        # Re-run einsum with the delta weight only (linear, so additive)
        # Extract einsum string from the wrapped module
        ein_str  = self.einsum.einsum_str
        lora_out = jnp.einsum(ein_str, lora_x, delta_w.astype(x.dtype))
        return base + self.scale * lora_out


class LinearLoRALayer(nnx.Module):
    """
    LoRA wrapper for nnx.Linear modules.
    Adds dropout to the LoRA branch.
    """
    def __init__(self, linear_module, rank: int, alpha: float, dropout: float,
                 dtype=jnp.bfloat16, rngs=None):
        self.linear   = linear_module
        self.rank     = rank
        self.scale    = alpha / rank
        self.dtype    = dtype
        self.dropout  = nnx.Dropout(dropout, rngs=rngs)
        
        in_dim  = linear_module.in_features
        out_dim = linear_module.out_features
        
        key_a, key_b = jax.random.split(rngs.params() if rngs else jax.random.PRNGKey(0))
        a_val = jax.random.normal(key_a, (in_dim, rank), dtype=dtype) * 0.01
        b_val = jnp.zeros((rank, out_dim), dtype=dtype)
        self.lora_a = nnx.LoRAParam(a_val)
        self.lora_b = nnx.LoRAParam(b_val)

    def __call__(self, x):
        lora_out = (self.dropout(x) @ self.lora_a[...]) @ self.lora_b[...]
        return self.linear(x) + self.scale * lora_out


def apply_lora_to_all(model, rank: int, alpha: float, dropout: float):
    """
    Apply LoRA with Dropout to:
      - nnx.Linear  modules  (MLP: gate_proj, up_proj, down_proj)
      - nnx.Einsum  modules  (Attention: q_einsum, kv_einsum, attn_vec_einsum)
    Returns the count of patched layers.
    """
    replaced = 0
    for path, module in nnx.graph.iter_graph(model):
        if not path:
            continue
        parent = model
        for step in path[:-1]:
            parent = getattr(parent, str(step))
        attr = str(path[-1])

        if isinstance(module, nnx.Linear):
            lora = LinearLoRALayer(
                module, rank=rank, alpha=alpha, dropout=dropout,
                dtype=jnp.bfloat16, rngs=nnx.Rngs(params=0, dropout=0),
            )
            setattr(parent, attr, lora)
            replaced += 1

        elif isinstance(module, nnx.Einsum):
            lora = EinsumLoRALayer(
                module, rank=rank, alpha=alpha, dropout=dropout,
                dtype=jnp.bfloat16, rngs=nnx.Rngs(params=0, dropout=0),
            )
            setattr(parent, attr, lora)
            replaced += 1

    return replaced


with mesh:
    n_lora = apply_lora_to_all(
        model, params.LORA_RANK, params.LORA_ALPHA, params.LORA_DROPOUT
    )

print(f"LoRA applied to {n_lora} layers (Linear + Einsum).")

# ── verify ────────────────────────────────────────────────────────────────────
leaves, _ = jax.tree_util.tree_flatten_with_path(nnx.state(model, nnx.LoRAParam))
lora_count = len(leaves)
print(f"LoRA tensors in state : {lora_count}  (expected > 0)")
if lora_count == 0:
    raise RuntimeError("LoRA application failed — do not proceed to training.")

show_hbm_usage()


Mesh shape : (1, 1)  axes=(tp, fsdp)


Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Weights at  : /home/lmassaron/.cache/huggingface/hub/models--google--gemma-3-1b-it/snapshots/dcc83ea841ab6100d6b47a070329e1ba4cf78752
Building model (bfloat16) …
LoRA applied to 78 layers (Linear + Einsum).
LoRA tensors in state : 156  (expected > 0)


## 6 · Optimizer, Trainer & Hooks

### Define export_adapter

This cell defines the `export_adapter` function used by both:
- **`CleanProgressHook`** — called automatically during training whenever eval loss improves, saving the best adapter to disk.
- **Manual export** — call `export_adapter(model, params)` at any time to inspect or re-export the current state.

In [6]:
import json
import safetensors.numpy as safe_np


def export_adapter(model, params):
    """
    Export all LoRAParam tensors as a PEFT-compatible adapter and save to
    params.OUTPUT_DIR.  Called automatically by CleanProgressHook whenever
    eval loss improves, so the files on disk always reflect the best checkpoint.
    Can also be called manually after training if needed.

    Convention reminders:
    - nnx.Linear LoRA: a.value=(in,rank), b.value=(rank,out)
      → PEFT lora_A.weight=(rank,in), lora_B.weight=(out,rank)  [both .T]
    - EinsumLoRALayer: lora_a=(in_flat,rank), lora_b=(rank,out_last)
    - DO NOT pre-scale B — PEFT applies alpha/rank itself.
    """
    lora_leaves, _ = jax.tree_util.tree_flatten_with_path(
        nnx.state(model, nnx.LoRAParam)
    )
    if not lora_leaves:
        print("\u26a0  No LoRAParam tensors found \u2014 was LoRA applied?")
        return

    def _key_str(k):
        s = str(k)
        for old, new in [("['", " "), ("']", " "), ("[", " "), ("]", " "), (".", " ")]:
            s = s.replace(old, new)
        return s.strip()

    MLP_MODULES     = {"gate_proj", "up_proj", "down_proj"}
    ATTN_EINSUMS    = {"q_einsum", "kv_einsum", "attn_vec_einsum"}
    ALL_TARGET_MODS = MLP_MODULES | ATTN_EINSUMS
    peft_state      = {}

    for key_path, val in lora_leaves:
        parts       = [_key_str(k) for k in key_path]
        parts_lower = [p.lower() for p in parts]

        layer_idx = next((p for p in parts if p.isdigit()), None)
        if layer_idx is None:
            continue

        mod = next((p for p in parts_lower if p in ALL_TARGET_MODS), None)
        if mod is None:
            continue

        last = parts_lower[-1] if parts_lower[-1] != "value" else parts_lower[-2]
        if last in ("lora_a", "a"):
            lora_part = "lora_A"
        elif last in ("lora_b", "b"):
            lora_part = "lora_B"
        else:
            continue

        weight = np.array(val, dtype=np.float32).T

        if mod in MLP_MODULES:
            key = (
                f"base_model.model.model.layers.{layer_idx}"
                f".mlp.{mod}.{lora_part}.weight"
            )
        else:
            einsum_map = {
                "q_einsum"        : "q_proj",
                "kv_einsum"       : "kv_proj",
                "attn_vec_einsum" : "o_proj",
            }
            hf_mod = einsum_map.get(mod, mod)
            key = (
                f"base_model.model.model.layers.{layer_idx}"
                f".self_attn.{hf_mod}.{lora_part}.weight"
            )
        peft_state[key] = weight

    n = len(peft_state)
    if n == 0:
        print("\u26a0  Path parsing found nothing.")
        return

    mlp_targets  = ["gate_proj", "up_proj", "down_proj"]

    os.makedirs(params.OUTPUT_DIR, exist_ok=True)
    weights_path = os.path.join(params.OUTPUT_DIR, "adapter_model.safetensors")
    config_path  = os.path.join(params.OUTPUT_DIR, "adapter_config.json")

    safe_np.save_file(peft_state, weights_path)

    adapter_cfg = {
        "base_model_name_or_path": params.MODEL_NAME,
        "peft_type"              : "LORA",
        "task_type"              : "CAUSAL_LM",
        "r"                      : params.LORA_RANK,
        "lora_alpha"             : params.LORA_ALPHA,
        "lora_dropout"           : params.LORA_DROPOUT,
        "target_modules"         : mlp_targets + ["q_proj", "kv_proj", "o_proj"],
        "bias"                   : "none",
    }
    with open(config_path, "w") as f:
        json.dump(adapter_cfg, f, indent=2)

    print(f"Adapter ({n} tensors) saved \u2192 {params.OUTPUT_DIR}/")


# ── Define export_adapter before running Cell 6 if starting fresh ─────────────
# This cell only defines the function.  The hook in Cell 6 references it by
# name, so this cell must be executed before Cell 6 on a fresh kernel.
# Execution order: Cell 8 (define) → Cell 6 (hook) → Cell 7 (train).
print("export_adapter defined.")


export_adapter defined.


In [ ]:
import optax
import time
from tunix.sft import hooks


# ── model-input builder ───────────────────────────────────────────────────────
def gen_model_input(inputs):
    padding_mask = jnp.where(inputs.input_tokens == 0, 0, 1)
    return {
        "input_tokens"  : inputs.input_tokens,
        "input_mask"    : inputs.input_mask,
        "positions"     : sft_utils.build_positions_from_mask(padding_mask),
        "attention_mask": sft_utils.make_causal_attn_mask(padding_mask),
    }


# ── progress + best-model hook ────────────────────────────────────────────────
class CleanProgressHook(hooks.TrainingHooks):
    """
    Logs every LOG_EVERY_N_STEPS steps.
    Tracks best eval loss and exports the adapter to disk whenever it improves,
    so the final adapter on disk is always the best checkpoint — not the last.
    """

    def __init__(self, log_every: int, max_steps: int,
                 model, params, export_fn, patience: int = 5):
        self._log_every  = log_every
        self._max_steps  = max_steps
        self._model      = model        # reference to the live JAX model
        self._params     = params
        self._export_fn  = export_fn    # export_adapter function (defined in Cell 8)
        self._t0         = None
        self._best_loss  = float("inf")
        self._best_step  = None
        self._patience   = patience
        self._no_improve_count = 0

    def on_train_start(self, train_ctx):
        self._t0 = time.time()
        print(f"{'Step':>6}  {'Loss':>9}  {'Perplexity':>11}  {'Elapsed':>8}")
        print("-" * 44)

    def on_train_end(self, train_ctx):
        elapsed = time.time() - self._t0
        print("-" * 44)
        print(f"Training complete in {elapsed:.1f}s")
        if self._best_step is not None:
            print(f"Best eval loss : {self._best_loss:.4f}  (at step {self._best_step})")
            print(f"Best adapter   : {self._params.OUTPUT_DIR}/ (already saved)")

    def on_train_step_start(self, train_ctx): pass

    def on_train_step_end(self, train_ctx, train_step, train_loss, step_time):
        if train_step % self._log_every == 0 or train_step == self._max_steps:
            elapsed = time.time() - self._t0
            ppl     = float(jnp.exp(jnp.minimum(train_loss, 20.0)))
            print(
                f"{train_step:>6}/{self._max_steps}  "
                f"{train_loss:>9.4f}  "
                f"{ppl:>11.2f}  "
                f"{elapsed:>7.1f}s",
                flush=True,
            )

    def on_eval_step_start(self, train_ctx): pass

    def on_eval_step_end(self, train_ctx, eval_loss):
        # Tunix reports eval loss as the SUM over all eval batches, not the mean.
        n_eval_batches = max(1, len(eval_idx) // params.BATCH_SIZE)
        mean_eval_loss = eval_loss / n_eval_batches
        ppl = float(jnp.exp(jnp.minimum(mean_eval_loss, 20.0)))

        step = getattr(train_ctx, "iter_steps", 0)
        marker = ""
        if mean_eval_loss < self._best_loss:
            self._best_loss        = mean_eval_loss
            self._best_step        = step // params.BATCH_SIZE
            self._no_improve_count = 0
            self._export_fn(self._model, self._params)
            marker = "  ← best, adapter saved"
        else:
            self._no_improve_count += 1
            if self._no_improve_count >= self._patience:
                print(f"  Early stopping: no improvement for {self._patience} evals.")
                # Signal trainer to stop — raise StopIteration or set a flag
                raise StopIteration  # PeftTrainer catches this and ends training

        print(f"  [eval]  mean_loss={mean_eval_loss:.4f}  ppl={ppl:.2f}  "
            f"(no-improve: {self._no_improve_count}/{self._patience}){marker}",
            flush=True)


# ── compute steps ─────────────────────────────────────────────────────────────
steps_per_epoch  = len(train_idx) // params.BATCH_SIZE
params.MAX_STEPS = steps_per_epoch * params.NUM_EPOCHS
warmup_steps     = int(params.MAX_STEPS * params.WARMUP_RATIO)

print(f"  Steps/epoch  : {steps_per_epoch:,}")
print(f"  Total steps  : {params.MAX_STEPS:,}  ({params.NUM_EPOCHS} epochs)")
print(f"  Warmup steps : {warmup_steps}")

# ── scheduler + optimizer ─────────────────────────────────────────────────────
schedule = optax.warmup_cosine_decay_schedule(
    init_value   = 0.0,
    peak_value   = params.LEARNING_RATE,
    warmup_steps = warmup_steps,
    decay_steps  = params.MAX_STEPS,
    end_value    = params.LEARNING_RATE * 0.05,
)
optimizer = optax.adamw(learning_rate=schedule, weight_decay=params.WEIGHT_DECAY)

# ── output directory ──────────────────────────────────────────────────────────
if os.path.exists(params.OUTPUT_DIR):
    shutil.rmtree(params.OUTPUT_DIR)
os.makedirs(params.OUTPUT_DIR, exist_ok=True)

# ── trainer ───────────────────────────────────────────────────────────────────
training_config = peft_trainer.TrainingConfig(
    max_steps                   = params.MAX_STEPS,
    eval_every_n_steps          = params.EVAL_EVERY_N_STEPS,
    gradient_accumulation_steps = params.GRAD_ACCUM_STEPS,
    checkpoint_root_directory   = os.path.abspath(params.OUTPUT_DIR),
)

trainer = (
    peft_trainer.PeftTrainer(
        model=model, optimizer=optimizer, training_config=training_config
    ).with_gen_model_input_fn(gen_model_input)
)

# NOTE: the hook is instantiated AFTER export_adapter is defined (Cell 8 must
# run first on a fresh kernel).  On re-runs within a session export_adapter
# is already in scope.  If you see a NameError, run Cell 8 once first, then
# re-run this cell and Cell 7.
trainer.with_training_hooks(
    CleanProgressHook(
        log_every  = params.LOG_EVERY_N_STEPS,
        max_steps  = params.MAX_STEPS,
        model      = model,
        params     = params,
        export_fn  = export_adapter,   # defined in Cell 8
    )
)

print("Trainer configured.")


  Steps/epoch  : 2,366
  Total steps  : 2,366  (1 epochs)
  Warmup steps : 118
Trainer configured.


## 7 · Train

Evaluation is run **by Tunix's `PeftTrainer`** internally every `EVAL_EVERY_N_STEPS` steps — the hook only receives the results via `on_eval_step_end`.  For this to work, the eval iterator must be **re-iterable** (see `ReusableEvalIter` below).

In [9]:
# ── Setting iterators ─────────────────────────────────────────────────────────

class ReusableEvalIter:
    """Re-iterable wrapper: returns a fresh eval generator on each iteration."""
    def __iter__(self):
        return make_data_iterator(
            full_ds, eval_idx, params.BATCH_SIZE,
            shuffle=False, infinite=False,
        )

# Train iterator stays a plain infinite generator — Tunix stops it via MAX_STEPS.
train_iter = make_data_iterator(
    full_ds, train_idx, params.BATCH_SIZE, shuffle=True, infinite=True
)
eval_iter = ReusableEvalIter()   # each eval pass gets fresh data

# ── Fine-tuning ───────────────────────────────────────────────────────────────

trainer.train(train_iter, eval_iter)


Adapter (156 tensors) saved → tunix-medical-model/
  [eval]  mean_loss=10.0764  ppl=23774.46  (no-improve: 0/5)  ← best, adapter saved


Training:   0%|          | 0/2366 [00:00<?, ?step/s]

  Step       Loss   Perplexity   Elapsed
--------------------------------------------
Adapter (156 tensors) saved → tunix-medical-model/
  [eval]  mean_loss=1.7300  ppl=5.64  (no-improve: 0/5)  ← best, adapter saved
   250/2366     1.8938         6.64    758.3s
Adapter (156 tensors) saved → tunix-medical-model/
  [eval]  mean_loss=1.6470  ppl=5.19  (no-improve: 0/5)  ← best, adapter saved
   500/2366     1.9904         7.32   1408.7s
Adapter (156 tensors) saved → tunix-medical-model/
  [eval]  mean_loss=1.6268  ppl=5.09  (no-improve: 0/5)  ← best, adapter saved
   750/2366     1.7306         5.64   2059.7s
Adapter (156 tensors) saved → tunix-medical-model/
  [eval]  mean_loss=1.5934  ppl=4.92  (no-improve: 0/5)  ← best, adapter saved
  1000/2366     1.4437         4.24   2710.0s
  [eval]  mean_loss=1.6221  ppl=5.06  (no-improve: 1/5)
  1250/2366     0.9219         2.51   3360.6s
  [eval]  mean_loss=1.5991  ppl=4.95  (no-improve: 2/5)
  1500/2366     1.3472         3.85   4011.0s
Adapte

In [10]:
from huggingface_hub import HfApi, create_repo


def push_adapter_to_hub(params, repo_id: str, private: bool = True):
    """
    Push the locally saved adapter to Hugging Face Hub.
    repo_id format: "your-username/your-repo-name"
    """
    api = HfApi()

    # Create the repo if it doesn't exist yet
    create_repo(repo_id, private=private, exist_ok=True)
    print(f"Repo ready : https://huggingface.co/{repo_id}")

    # Upload the two adapter files
    for filename in ("adapter_model.safetensors", "adapter_config.json"):
        local_path = os.path.join(params.OUTPUT_DIR, filename)
        if not os.path.exists(local_path):
            print(f"⚠  {filename} not found — run the export cell first.")
            return
        api.upload_file(
            path_or_fileobj=local_path,
            path_in_repo=filename,
            repo_id=repo_id,
        )
        print(f"  Uploaded : {filename}")

    # Optionally write a minimal README so the model card is not empty
    readme = f"""---
base_model: {params.MODEL_NAME}
library_name: peft
---

# {repo_id.split("/")[-1]}

LoRA adapter fine-tuned from `{params.MODEL_NAME}` on a medical cardiology QA dataset
using [Tunix](https://github.com/google/tunix) (JAX/Flax).

## Usage

```python
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base  = AutoModelForCausalLM.from_pretrained("{params.MODEL_NAME}")
model = PeftModel.from_pretrained(base, "{repo_id}")
tok   = AutoTokenizer.from_pretrained("{params.MODEL_NAME}")
```
"""
    api.upload_file(
        path_or_fileobj=readme.encode(),
        path_in_repo="README.md",
        repo_id=repo_id,
    )
    print("  Uploaded : README.md")
    print(f"\nDone → https://huggingface.co/{repo_id}")


# ── set your repo id and push ─────────────────────────────────────────────────
HF_REPO_ID = "lmassaron/gemma-3-1b-medical-cardiology-lora"
push_adapter_to_hub(params, repo_id=HF_REPO_ID, private=False)

Repo ready : https://huggingface.co/lmassaron/gemma-3-1b-medical-cardiology-lora


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

  Uploaded : adapter_model.safetensors
  Uploaded : adapter_config.json


No files have been modified since last commit. Skipping to prevent empty commit.


  Uploaded : README.md

Done → https://huggingface.co/lmassaron/gemma-3-1b-medical-cardiology-lora
